# music2latent

In [1]:
# basic imports
import numpy as np
import IPython
import librosa
import os 
os.environ["CUDA_VISIBLE_DEVICES"]="-1"

Initialize the EncoderDecoder model.

In [ ]:
from music2latent import EncoderDecoder
from music2latent.transforms import StreamableSTFT

from music2latent.config_loader import load_config


config = "/data/nils/repos/codecs_benchmark/music2latent/configs/config_drums_2048.py"
load_config(config)

transform = StreamableSTFT(nfft = 1024, hop_size = 256, skip_features = 1)

encdec = EncoderDecoder(load_path_inference = "/data/nils/repos/codecs_benchmark/music2latent/checkpoints/2025-05-28 12:58:26.571568/model_fid_13.808908367473272_loss_26.269_iters_958342.pt",transform = transform )

Let's load an audio file for this tutorial:

In [3]:
# audio_path = librosa.example('trumpet')

# wv, sr = librosa.load(audio_path, sr=44100)

# IPython.display.display(IPython.display.Audio(wv, rate=sr))

## Compare representations

In [3]:

import numpy as np
import matplotlib.pyplot as plt

def plot_spectrograms_amplitude_phase(spectrograms, titles=None, cmap_amplitude='magma', cmap_phase='twilight', figsize=(15, 6)):
    """
    Plots amplitude and phase of spectrograms from real/imag format.

    Parameters:
    - spectrograms: list of np.ndarray with shape (2, F, T), where 2 = (real, imag).
    - titles: optional list of titles for each spectrogram.
    - cmap_amplitude: colormap for amplitude.
    - cmap_phase: colormap for phase.
    - figsize: base figure size.
    """
    spectrograms = [s.squeeze().cpu().detach().numpy() for s in spectrograms]
    n = len(spectrograms)
    fig, axes = plt.subplots(2, n, figsize=(figsize[0], figsize[1]), constrained_layout=True)
    
    # Ensure axes are 2D for consistent indexing
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, spec in enumerate(spectrograms):
        if spec.shape[0] != 2:
            raise ValueError(f"Spectrogram {i} must have shape (2, F, T), got {spec.shape}")
        
        # Reconstruct complex spectrogram
        complex_spec = spec[0] + 1j * spec[1]
        amp = np.log(np.abs(complex_spec))
        phase = np.angle(complex_spec)

        ax_amp = axes[0, i]
        im_amp = ax_amp.imshow(amp, origin='lower', aspect='auto', cmap=cmap_amplitude)
        ax_amp.set_title(titles[i] if titles else f'Spectrogram {i+1}', fontsize=12, weight='bold')
        ax_amp.set_ylabel('Amplitude', fontsize=10)
        ax_amp.set_xlabel('Time', fontsize=10)
        fig.colorbar(im_amp, ax=ax_amp, fraction=0.046, pad=0.04)

        ax_phase = axes[1, i]
        im_phase = ax_phase.imshow(phase, origin='lower', aspect='auto', cmap=cmap_phase)
        ax_phase.set_ylabel('Phase', fontsize=10)
        ax_phase.set_xlabel('Time', fontsize=10)
        fig.colorbar(im_phase, ax=ax_phase, fraction=0.046, pad=0.04)

    plt.suptitle('Amplitude and Phase of Complex Spectrograms', fontsize=16, weight='bold')
    plt.show()


In [ ]:
import torch

audio_path ="/data/nils/datasets/drums/breaks/audio/breaksv1/DNB_BREAK_04.wav"
wv, sr = librosa.load(audio_path, sr=44100)
wv = torch.from_numpy(wv).reshape(1,1,-1)

transform = StreamableSTFT(nfft = 1024, hop_size = 256, skip_features = 1)

S1 = transform.forward(wv)

from music2latent.audio import wv2realimag
S2 = wv2realimag(wv.squeeze(1), 512)

print(S1.shape, S2.shape)

In [ ]:
plot_spectrograms_amplitude_phase([S1, S2])

# Encode

To encode an audio sample into latents, you need to provide the waveform as input, with shape [audio_channels, waveform_samples] or simply [waveform_samples,]:

In [ ]:
latent

In [ ]:
audio_path ="/data/nils/datasets/drums/darbouka/audio/darbouka-comp/darbouka-comp-012.wav"
wv, sr = librosa.load(audio_path, sr=44100, duration = 12)

print('Original')
IPython.display.display(IPython.display.Audio(wv, rate=sr))

# wv=  np.linspace(0, 1, 131072)
# print(f'waveform samples: {wv.shape}')

latent = encdec.encode(wv)
print(f'Shape of latents: {latent.shape}')

You can also process a batch of waveforms. Just use a numpy array with shape [batch_size, waveform_samples] as input:

# Decode

To decode latent embeddings back to waveform, be sure to have latents with shape [batch_size/audio_channels, latent_dim, latent_length]:

In [ ]:
for nb_steps in [1, 3, 5]:
    wv_rec = encdec.decode(latent, denoising_steps= nb_steps)
    print(f'Shape of decoded waveform: {wv_rec.shape}')

    print(wv.shape, wv_rec.shape)
    print('Reconstructed')
    IPython.display.display(IPython.display.Audio(wv_rec.squeeze().cpu().numpy(), rate=sr))

In [ ]:
import time

st = time.time()
a = encdec.encode(wv_batched[:1])
encdec.decode(a)    
et = time.time()

print(et-st)

In [ ]:
import torch
torch.jit.script(encdec.gen)

You can also specify how many denoising steps to perform (default is 1). However, we do not notice any improvements in audio quality by increasing the denoise_steps.

In [ ]:
wv_rec = encdec.decode(latent, denoising_steps=3)
print(f'Shape of decoded waveform: {wv_rec.shape}')

print('Original')
IPython.display.display(IPython.display.Audio(wv, rate=sr))
print('Reconstructed')
IPython.display.display(IPython.display.Audio(wv_rec.squeeze().cpu().numpy(), rate=sr))

# Keeping GPU memory under control

The autoencoder model needs plenty of memory to encode and decode samples.
We offer a way to keep the memory usage under control.

You can specify both the __max_batch_size__ and __max_waveform_length__ to use for encoding or decoding samples.

If not specified, the default values are the ones in hparams_inference.py (__max_batch_size__=1, __max_waveform_length__=44100*10)

If the waveform sample to encode or to reconstruct is longer than __max_waveform_length__, the spectrogram representation will be split into multiple samples, processed sequentially, and then concatenated back together.

In [ ]:
# wv, sr = librosa.load(audio_path, sr=44100)
# print(f'waveform samples: {wv.shape}')

# # split spectrogram into 1 second chunks, process each chunk separately, concatenate the results
# # much lower memory usage
# latent = encdec.encode(wv, max_waveform_length=44100*1)
# print(f'Shape of latents: {latent.shape}')

# wv_rec = encdec.decode(latent, max_waveform_length=44100*1)
# print(f'Shape of decoded waveform: {wv_rec.shape}')

# print('Original')
# IPython.display.display(IPython.display.Audio(wv, rate=sr))
# print('Reconstructed')
# IPython.display.display(IPython.display.Audio(wv_rec.squeeze().cpu().numpy(), rate=sr))

If you need to encode/decode batches of samples in parallel you can increase the __max_batch_size__ argument until you reach your maximum memory budget:

In [ ]:
# wv, sr = librosa.load(audio_path, sr=44100)

# # create a batch of waveforms
# wv_batched = np.stack([wv]*3, axis=0)
# print(f'batch of waveforms shape: {wv_batched.shape}')

# latent_batched = encdec.encode(wv_batched, max_batch_size=3)
# print(f'Shape of batched latents: {latent_batched.shape}')

# wv_rec = encdec.decode(latent, max_batch_size=3)
# print(f'Shape of decoded waveform: {wv_rec.shape}')

# print('Original')
# IPython.display.display(IPython.display.Audio(wv, rate=sr))
# print('Reconstructed')
# IPython.display.display(IPython.display.Audio(wv_rec[0].squeeze().cpu().numpy(), rate=sr))

# Keep in Mind:

When using the latents for generation tasks using diffusion-type models, make sure to properly normalize the latents according to the chosen diffusion framework. The latents extracted with this library are rescaled to have unit standard deviation for a reference music dataset, but ensure that the latents are properly normalized for your specific use case.